# Advanced Python Enum: Automatic Values — 24 Problems with Complete Solutions

A **runnable, test-first problem workbook** expanding the supplied lesson on `enum.auto()`, `_generate_next_value_()`, aliases, reusable enum bases, and intentionally opaque values. Contains 24 worked problems, extra recipes, and an integrated regression suite. All code uses the Python standard library; **Python 3.11+** is required (tested on Python 3.13). Run **Kernel → Restart & Run All**.

**Learning goals:** predict `auto()` across enum families; design custom generators; diagnose alias collisions; distinguish member names from values; manage compatibility at persistence/API boundaries; test deterministic generation; apply `unique` and `verify`; and avoid unbounded random generation.

**Method:** each problem includes a specification, runnable solution, assertions, an explanation, and a stretch exercise. Unlike merely looking at example output, tests fail loudly when an assumption is wrong. Identifiers are intentionally unique across cells for safe top-to-bottom execution.

## Important version notes and technical corrections

- The provided source lesson was authored for Python 3.6.7. Its core examples are preserved as the starting topic; this workbook adds modern, explicitly documented behavior.
- For plain `Enum` / `IntEnum`, `auto()` starts at **1** and chooses increasing integers. In **Python 3.13**, the default generator uses the **highest previous value plus one** (older versions could use the last value instead). Do **not** use implicit numeric enum values as external IDs or assume reordering does not change them.
- `Flag` / `IntFlag` use successive powers of two; `StrEnum` (Python 3.11+) uses lower-case names.
- `_generate_next_value_(name, start, count, last_values)` must appear **before members**. `count` includes previously declared names even if they are aliases; `last_values` contains previous assigned values, including duplicates.
- `@unique` **detects and rejects** aliases at class definition; it does not repair them. Intentional aliases are valid without `@unique`.
- Use `random.Random(seed)` for repeatable *examples*, not public identifiers or secrets. Opaque `object()` values discourage numeric coupling but do **not** enforce security.

Official reference: [Python enum library reference](https://docs.python.org/3/library/enum.html) · [Python Enum HOWTO](https://docs.python.org/3/howto/enum.html).

In [1]:
import sys
import random
import json
from collections import Counter
from enum import (
    Enum, IntEnum, StrEnum, Flag, IntFlag, auto, unique,
    verify, CONTINUOUS, NAMED_FLAGS,
)

assert sys.version_info >= (3, 11), "This workbook requires Python 3.11+."
print("Python:", sys.version.split()[0])
print("Imports successful; ready for 24 problems.")

Python: 3.13.7
Imports successful; ready for 24 problems.


## Problem 01 — Baseline automatic integers and identity

**Challenge.** Create a three-stage process, prove the default values and identity, and explain when comparing `.value` is inappropriate.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 01

`auto()` is evaluated while the class is constructed, not on each member access. Compare members (`is` or `==`) in application logic. Keep `.value` out of durable representations unless values are explicitly part of a stable contract.

In [2]:
@unique
class WorkflowStage01(Enum):
    WAITING = auto()
    RUNNING = auto()
    FINISHED = auto()

assert [(m.name, m.value) for m in WorkflowStage01] == [
    ("WAITING", 1), ("RUNNING", 2), ("FINISHED", 3)
]
assert WorkflowStage01.RUNNING is WorkflowStage01["RUNNING"]
assert WorkflowStage01(2) is WorkflowStage01.RUNNING
assert WorkflowStage01.WAITING is not WorkflowStage01.FINISHED
print([(member.name, member.value) for member in WorkflowStage01])

[('WAITING', 1), ('RUNNING', 2), ('FINISHED', 3)]


**Further practice:** Insert a member between `WAITING` and `RUNNING` and predict which generated integers change.

## Problem 02 — Find an accidental alias when manual values are mixed

**Challenge.** Replicate the original lesson’s subtle collision: `FIRST = auto()` followed by `SECOND = 1`. Prove that iteration silently hides one name and that `__members__` does not.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 02

An alias is another **name for an existing member**, not a new member. Enum iteration yields canonical members; `__members__` retains all declared names (and is read-only).

In [3]:
class Collision02(Enum):
    FIRST = auto()
    SECOND = 1
    THIRD = auto()

assert Collision02.SECOND is Collision02.FIRST
assert [m.name for m in Collision02] == ["FIRST", "THIRD"]
assert list(Collision02.__members__) == ["FIRST", "SECOND", "THIRD"]
assert Collision02.__members__["SECOND"] is Collision02.FIRST
print("Iterated:", [m.name for m in Collision02])
print("All declared names:", list(Collision02.__members__))

Iterated: ['FIRST', 'THIRD']
All declared names: ['FIRST', 'SECOND', 'THIRD']


**Further practice:** Try using `SECOND = 2`: predict what `THIRD = auto()` will do on your Python version.

## Problem 03 — Turn silent aliasing into an early validation error

**Challenge.** Show a reproducible test for a bad enum definition and a corrected enum that cannot accidentally contain aliases.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 03

Use `@unique` for enums in which every name must designate a distinct member. Catch its error only in tests or examples; normally let the import fail so the issue is fixed immediately.

In [4]:
try:
    @unique
    class Broken03(Enum):
        ONE = auto()
        DUPLICATE = 1
except ValueError as exc:
    assert "DUPLICATE" in str(exc)
    print("Caught expected class-definition error:", exc)
else:
    raise AssertionError("Expected @unique to reject the alias")

@unique
class Fixed03(Enum):
    ONE = auto()
    TWO = auto()
    THREE = auto()

assert len(Fixed03) == len(Fixed03.__members__) == 3

Caught expected class-definition error: duplicate values found in <enum 'Broken03'>: DUPLICATE -> ONE


**Further practice:** Explain why wrapping a production enum declaration in `try/except` and continuing would conceal a programming defect.

## Problem 04 — Audit all four generator parameters

**Challenge.** Instrument the generator to collect `name`, `start`, `count`, and `last_values`, including a deliberate alias. Determine whether aliases contribute to `count`.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 04

The generator sees declarations that precede each `auto()`, including the repeated value that becomes an alias. `count` is **declaration count** here, not count of distinct members. Assign values from the actual callback inputs rather than guessing.

In [5]:
audit04 = []

class Audited04(Enum):
    @staticmethod
    def _generate_next_value_(name, start, count, last_values):
        audit04.append((name, start, count, tuple(last_values)))
        return 100 + 10 * count

    FIRST = 7
    ALSO_FIRST = 7       # intentional alias
    SECOND = auto()
    THIRD = auto()

assert audit04 == [
    ("SECOND", 1, 2, (7, 7)),
    ("THIRD", 1, 3, (7, 7, 120)),
]
assert Audited04.SECOND.value == 120
assert Audited04.THIRD.value == 130
print("Generator calls:", audit04)

Generator calls: [('SECOND', 1, 2, (7, 7)), ('THIRD', 1, 3, (7, 7, 120))]


**Further practice:** Change the generator to `len(set(last_values))` and predict the second and third generated values.

## Problem 05 — Create a geometric progression without hidden defaults

**Challenge.** Implement `auto()` values 3, 9, 27, 81 using a custom generator; make the result independent of the implementation’s default numeric sequence.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 05

Define the algorithm before defining members. Since all declarations in this class use `auto()` and no aliases exist, `count` is a simple zero-based position. This would no longer be true after adding explicit alias declarations.

In [6]:
@unique
class PowersOfThree05(Enum):
    @staticmethod
    def _generate_next_value_(name, start, count, last_values):
        return 3 ** (count + 1)

    FIRST = auto()
    SECOND = auto()
    THIRD = auto()
    FOURTH = auto()

assert [m.value for m in PowersOfThree05] == [3, 9, 27, 81]
assert PowersOfThree05.FOURTH.value // PowersOfThree05.THIRD.value == 3
print([m.value for m in PowersOfThree05])

[3, 9, 27, 81]


**Further practice:** Write a generator for powers of 5, and test the first six values.

## Problem 06 — Avoid collisions with a deliberately mixed numeric enum

**Challenge.** Specify an algorithm that chooses a value larger than all preceding values when explicit values and generated values are mixed. Verify exact results rather than relying on version-dependent default behavior.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 06

An explicit generator documents exactly how mixed values behave. It prevents numeric collisions **for this particular monotone strategy** but does not make renumbering safe for a persistent schema; explicitly pin values in those contracts.

In [7]:
@unique
class SafeMixed06(Enum):
    @staticmethod
    def _generate_next_value_(name, start, count, last_values):
        return max(last_values, default=0) + 1

    LOW = 5
    NORMAL = auto()
    CUSTOM = 20
    HIGH = auto()
    LOWEST = -10
    HIGHEST = auto()

assert [m.value for m in SafeMixed06] == [5, 6, 20, 21, -10, 22]
assert len(set(m.value for m in SafeMixed06)) == len(SafeMixed06)
print({m.name: m.value for m in SafeMixed06})

{'LOW': 5, 'NORMAL': 6, 'CUSTOM': 20, 'HIGH': 21, 'LOWEST': -10, 'HIGHEST': 22}


**Further practice:** If the first explicit member were `-10`, adapt the `default` handling to support a different desired starting point.

## Problem 07 — Use functional Enum creation and its start argument

**Challenge.** Create enums from data without writing a class statement. Prove that the `start` parameter of the functional API changes the assigned starting integer.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 07

The functional API can create `Enum` from names and a `start`; for programmatic construction prefer controlled, fixed input. Do not build executable class code from untrusted input, and do not persist auto-assigned numbers casually.

In [8]:
Priority07 = Enum("Priority07", ["LOW", "MEDIUM", "HIGH"], start=10)
Status07 = Enum("Status07", "QUEUED RUNNING DONE", start=40)

assert [m.value for m in Priority07] == [10, 11, 12]
assert [m.value for m in Status07] == [40, 41, 42]
assert Priority07["MEDIUM"].value == 11
assert Priority07(12) is Priority07.HIGH
print("Functional enum:", list(Priority07))

Functional enum: [<Priority07.LOW: 10>, <Priority07.MEDIUM: 11>, <Priority07.HIGH: 12>]


**Further practice:** Create a five-member enum starting at 101 and test all five integers.

## Problem 08 — Derive readable title-case values

**Challenge.** Replace numeric auto values with title-case member names and prove value lookup and member-name lookup are distinct operations.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 08

The generator may return nonnumeric values. `Enum["NAME"]` resolves **names**, while `Enum(value)` resolves **values**. Human-readable labels can change for localization, so separate them from a protocol contract.

In [9]:
@unique
class HumanState08(Enum):
    @staticmethod
    def _generate_next_value_(name, start, count, last_values):
        return name.title().replace("_", " ")

    NOT_STARTED = auto()
    IN_PROGRESS = auto()
    COMPLETE = auto()

assert HumanState08.NOT_STARTED.value == "Not Started"
assert HumanState08.IN_PROGRESS.value == "In Progress"
assert HumanState08["COMPLETE"] is HumanState08.COMPLETE
assert HumanState08("Complete") is HumanState08.COMPLETE
print({m.name: m.value for m in HumanState08})

{'NOT_STARTED': 'Not Started', 'IN_PROGRESS': 'In Progress', 'COMPLETE': 'Complete'}


**Further practice:** Add `WAITING_FOR_REVIEW`; check whether your desired capitalization of acronyms needs a mapping.

## Problem 09 — Detect collisions caused by string normalization

**Challenge.** Demonstrate that two distinct enum identifiers can normalize to the same value, and make declaration fail instead of quietly creating an alias.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 09

`@unique` is useful even for name-derived generators: a normalization transformation need not be injective. Validate the completed class rather than trusting the transformation.

In [10]:
def normalize09(name: str) -> str:
    return name.lower().replace("_", "-")

try:
    @unique
    class BadSlug09(Enum):
        @staticmethod
        def _generate_next_value_(name, start, count, last_values):
            return normalize09(name)
        API_READY = auto()
        api_ready = auto()
except ValueError as exc:
    assert "api_ready" in str(exc)
    print("Normalization collision detected:", exc)
else:
    raise AssertionError("Expected @unique to catch normalized collision")

@unique
class GoodSlug09(Enum):
    @staticmethod
    def _generate_next_value_(name, start, count, last_values):
        return normalize09(name)
    API_READY = auto()
    API_FAILED = auto()

assert [m.value for m in GoodSlug09] == ["api-ready", "api-failed"]

Normalization collision detected: duplicate values found in <enum 'BadSlug09'>: api_ready -> API_READY


**Further practice:** Try a normalization that removes hyphens entirely and identify a distinct-name pair that collides.

## Problem 10 — Build a reusable generator-only Enum base

**Challenge.** Avoid copying `_generate_next_value_` into every enum by creating a base with **no actual members**, then derive two enums with independent namespaces.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 10

An enum without members can be subclassed to share behavior. A concrete enum with members normally cannot be extended by adding new members. Distinct enum classes keep their own member identity even when values coincide.

In [11]:
class LowercaseBase10(Enum):
    @staticmethod
    def _generate_next_value_(name, start, count, last_values):
        return name.lower()

@unique
class TaskKind10(LowercaseBase10):
    IMPORT = auto()
    EXPORT = auto()

@unique
class ResourceKind10(LowercaseBase10):
    FILE = auto()
    FOLDER = auto()

assert TaskKind10.IMPORT.value == "import"
assert ResourceKind10.FOLDER.value == "folder"
assert TaskKind10.IMPORT != ResourceKind10.FILE
print(TaskKind10.IMPORT, ResourceKind10.FILE)

TaskKind10.IMPORT ResourceKind10.FILE


**Further practice:** Add a third subclass with four members; verify `@unique` and correct lowercase strings.

## Problem 11 — Choose StrEnum for string-like interoperability

**Challenge.** Compare a custom string-valued `Enum` with `StrEnum` and show that the latter has lowercase `auto()` by default while retaining enum identity.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 11

`StrEnum` was introduced in 3.11. Use it when actual string behavior is needed; use plain `Enum` when values should not compare equal to raw strings. Some third-party code requires `type(x) is str`: explicitly convert via `str(member)` in such cases.

In [12]:
@unique
class WireState11(StrEnum):
    READY = auto()
    IN_PROGRESS = auto()
    DONE = auto()

assert WireState11.IN_PROGRESS.value == "in_progress"
assert WireState11.READY == "ready"  # StrEnum interoperates with str
assert isinstance(WireState11.READY, str)
assert WireState11["DONE"] is WireState11.DONE
assert str(WireState11.DONE) == "done"

@unique
class PlainStringState11(Enum):
    @staticmethod
    def _generate_next_value_(name, start, count, last_values):
        return name.lower()
    READY = auto()

assert PlainStringState11.READY != "ready"
assert PlainStringState11.READY.value == "ready"
print("StrEnum:", WireState11.READY, "Plain Enum:", PlainStringState11.READY)

StrEnum: ready Plain Enum: PlainStringState11.READY


**Further practice:** Test JSON serialization with `json.dumps` for `StrEnum` and for an ordinary string-valued `Enum`.

## Problem 12 — Show why IntEnum arithmetic loses the enum type

**Challenge.** Represent legacy numeric codes with `IntEnum`, then prove that arithmetic produces an `int` and can accidentally mix domains.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 12

`IntEnum` is suitable when integer interoperability is deliberate. It is **not** arithmetic type safety: operations strip enum status, and equal integers can mix with unrelated domains.

In [13]:
@unique
class LegacyCode12(IntEnum):
    SUCCESS = auto()
    RETRY = auto()
    FAILURE = auto()

assert LegacyCode12.SUCCESS == 1
assert LegacyCode12.SUCCESS == 1  # unlike plain Enum
result12 = LegacyCode12.SUCCESS + LegacyCode12.RETRY
assert result12 == 3
assert type(result12) is int
assert not isinstance(result12, LegacyCode12)
assert LegacyCode12.FAILURE.value == result12  # same number, different meaning
print("Arithmetic result and type:", result12, type(result12).__name__)

Arithmetic result and type: 3 int


**Further practice:** Write `def is_success(code: LegacyCode12) -> bool` using enum identity, and explain runtime type validation.

## Problem 13 — Generate non-overlapping Flag bits

**Challenge.** Define independent permissions and combine them safely. Compare the resulting bits with ordinary sequential `Enum` integers.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 13

`Flag` / `IntFlag` assign powers of two rather than 1, 2, 3, ... because each bit must represent an independent capability. Test combinations with bitwise operators, not addition.

In [14]:
@unique
class Permission13(Flag):
    READ = auto()
    WRITE = auto()
    EXECUTE = auto()
    SHARE = auto()

assert [m.value for m in Permission13] == [1, 2, 4, 8]
combined13 = Permission13.READ | Permission13.EXECUTE
assert combined13.value == 5
assert combined13 & Permission13.READ == Permission13.READ
assert combined13 & Permission13.WRITE == Permission13(0)
assert combined13 | Permission13.WRITE == (
    Permission13.READ | Permission13.WRITE | Permission13.EXECUTE
)
print("Permission bits:", [(m.name, m.value) for m in Permission13])

Permission bits: [('READ', 1), ('WRITE', 2), ('EXECUTE', 4), ('SHARE', 8)]


**Further practice:** Construct `READ | WRITE | SHARE`, then test the presence and absence of each base permission.

## Problem 14 — Handle named composite flags and alias introspection

**Challenge.** Add a named composite permission and explain why a named composite is not an independently allocated base bit.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 14

Named composites are legitimate for flags and appear in `__members__`. Iteration normally returns single-bit canonical members; use `__members__` to inspect all names. Do not insist on `@unique` if intentional aliases are part of the design.

In [15]:
class NamedPermission14(Flag):
    READ = auto()
    WRITE = auto()
    EXECUTE = auto()
    READ_WRITE = READ | WRITE

assert NamedPermission14.READ_WRITE.value == 3
assert NamedPermission14.READ_WRITE == (
    NamedPermission14.READ | NamedPermission14.WRITE
)
assert [m.name for m in NamedPermission14] == ["READ", "WRITE", "EXECUTE"]
assert "READ_WRITE" in NamedPermission14.__members__
assert list(NamedPermission14.__members__) == [
    "READ", "WRITE", "EXECUTE", "READ_WRITE"
]
print("Named composite:", NamedPermission14.READ_WRITE)

Named composite: NamedPermission14.READ_WRITE


**Further practice:** Add `ALL = READ | WRITE | EXECUTE` and confirm it is excluded from iteration.

## Problem 15 — Use verification decorators for integer continuity

**Challenge.** Reject a numeric enum with missing values using `@verify(CONTINUOUS)`, then create a valid contiguous enum.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 15

`verify(CONTINUOUS)` (Python 3.11+) enforces no missing integer value between the minimum and maximum. It is useful for deliberately contiguous codes, not for sparse wire protocols or flags.

In [16]:
try:
    @verify(CONTINUOUS)
    class Gapped15(Enum):
        FIRST = 1
        THIRD = 3
except ValueError as exc:
    assert "2" in str(exc)
    print("Detected expected gap:", exc)
else:
    raise AssertionError("CONTINUOUS should reject missing value 2")

@verify(CONTINUOUS)
@unique
class Continuous15(Enum):
    FIRST = 5
    SECOND = auto()
    THIRD = auto()

assert [m.value for m in Continuous15] == [5, 6, 7]

Detected expected gap: invalid enum 'Gapped15': missing values 2


**Further practice:** Explain why a sparse enumeration with valid reserved protocol code ranges should not use `CONTINUOUS`.

## Problem 16 — Reject improperly composed flag aliases

**Challenge.** Use `verify(NAMED_FLAGS)` to detect a composite which includes an unnamed bit; then fix the definition.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 16

`NAMED_FLAGS` validates that bits in named combined flags are named individually. It does not mean every possible bit combination needs a declared name. Keep base bits explicit and separate.

In [17]:
try:
    @verify(NAMED_FLAGS)
    class InvalidMask16(Flag):
        READ = 1
        WRITE = 2
        BROKEN = 1 | 4    # bit 4 is never named individually
except ValueError as exc:
    assert "BROKEN" in str(exc)
    print("Caught unnamed-flag bit:", str(exc).splitlines()[0])
else:
    raise AssertionError("NAMED_FLAGS should reject the unnamed bit")

@verify(NAMED_FLAGS)
class ValidMask16(Flag):
    READ = auto()
    WRITE = auto()
    EXECUTE = auto()
    READ_EXECUTE = READ | EXECUTE

assert ValidMask16.READ_EXECUTE.value == 5

Caught unnamed-flag bit: invalid Flag 'InvalidMask16': alias BROKEN is missing value 0x4 [use enum.show_flag_values(value) for details]


**Further practice:** Add `ALL` by OR-ing every named base bit; verify class creation succeeds.

## Problem 17 — Generate bounded deterministic unique random values

**Challenge.** Build a seeded local-random factory with a finite value pool, exact reproducibility, and an explicit exhaustion error rather than an infinite `while True` loop.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 17

A finite pool makes termination and uniqueness explicit. A local RNG avoids contaminating other notebook cells. Seeded random codes are **not** unpredictable, cryptographically secure, or suitable as persistent/public identifiers.

In [18]:
def random_code_enum17(seed: int, low: int, high: int):
    rng = random.Random(seed)  # isolated: does not modify global RNG

    @unique
    class RandomCodes17(Enum):
        @staticmethod
        def _generate_next_value_(name, start, count, last_values):
            available = [v for v in range(low, high + 1)
                         if v not in last_values]
            if not available:
                raise ValueError("Not enough distinct codes in configured range")
            return rng.choice(available)

        ALPHA = auto()
        BETA = auto()
        GAMMA = auto()
        DELTA = auto()

    return RandomCodes17

r1_17 = random_code_enum17(2026, 10, 20)
r2_17 = random_code_enum17(2026, 10, 20)
values17 = [m.value for m in r1_17]
assert values17 == [m.value for m in r2_17]
assert len(values17) == len(set(values17)) == 4
assert all(10 <= value <= 20 for value in values17)
print("Deterministic example codes:", values17)

Deterministic example codes: [11, 16, 20, 12]


**Further practice:** Adapt the factory to accept `n` member names with the functional `Enum` API, or prove exhaustion in a class with five members and a pool of four integers.

## Problem 18 — Test exhaustion without hanging the notebook

**Challenge.** Build a tiny-range generator with insufficient capacity. Assert that the failure occurs at class creation and does not loop forever.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 18

A `while True` generator can become non-terminating when all unique values are used. Compute remaining capacity first and raise a clear exception if none remains.

In [19]:
def construct_too_small18():
    rng = random.Random(1)

    class Tiny18(Enum):
        @staticmethod
        def _generate_next_value_(name, start, count, last_values):
            choices = [n for n in (10, 11) if n not in last_values]
            if not choices:
                raise ValueError("Code space exhausted")
            return rng.choice(choices)

        FIRST = auto()
        SECOND = auto()
        THIRD = auto()

    return Tiny18

try:
    construct_too_small18()
except ValueError as exc:
    assert str(exc) == "Code space exhausted"
    print("Expected bounded failure:", exc)
else:
    raise AssertionError("Expected exhaustion to fail promptly")

Expected bounded failure: Code space exhausted


**Further practice:** Calculate the maximum possible members for an integer range of `low=100, high=109`.

## Problem 19 — Use opaque member values without inventing false security

**Challenge.** Model statuses whose associated numeric value is meaningless. Generate fresh object sentinels automatically, assert uniqueness, and serialize by name instead of object value.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 19

Unique object sentinels make the `.value` awkward to use as a literal. This is a design cue, **not** access control or a security boundary. Values are process-local and unsuitable for external storage.

In [20]:
class OpaqueBase19(Enum):
    @staticmethod
    def _generate_next_value_(name, start, count, last_values):
        return object()

@unique
class JobState19(OpaqueBase19):
    QUEUED = auto()
    RUNNING = auto()
    DONE = auto()

assert JobState19.QUEUED.value is JobState19.QUEUED.value
assert JobState19.QUEUED.value is not JobState19.RUNNING.value
assert len(JobState19) == 3

wire_name19 = JobState19.RUNNING.name
assert JobState19[wire_name19] is JobState19.RUNNING
try:
    json.dumps(JobState19.RUNNING.value)
except TypeError:
    pass
else:
    raise AssertionError("An opaque object should not JSON-serialize directly")
print("Safe symbolic encoding:", wire_name19)

Safe symbolic encoding: RUNNING


**Further practice:** Add an enum `ErrorKind19` deriving from the same base; prove members from different classes remain distinct.

## Problem 20 — Preserve durable meaning across enum evolution

**Challenge.** Demonstrate that inserting an `auto()` member changes numeric values but name-based storage can still represent the same conceptual member.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 20

Implicit numbers are an internal implementation detail unless deliberately frozen. Names survive insertion *if names are kept stable*. For a wire format or database, explicitly assigned, versioned values are another valid choice.

In [21]:
class VersionOne20(Enum):
    QUEUED = auto()
    RUNNING = auto()
    DONE = auto()

class VersionTwo20(Enum):
    QUEUED = auto()
    PAUSED = auto()    # inserted after QUEUED
    RUNNING = auto()
    DONE = auto()

assert VersionOne20.RUNNING.value == 2
assert VersionTwo20.RUNNING.value == 3
old_numeric20 = VersionOne20.RUNNING.value
assert VersionTwo20(old_numeric20) is VersionTwo20.PAUSED  # WRONG meaning

stored20 = {"state": VersionOne20.RUNNING.name}
restored20 = VersionTwo20[stored20["state"]]
assert restored20 is VersionTwo20.RUNNING
print("Old/new numbers:", old_numeric20, restored20.value,
      "safe stored name:", stored20["state"])

Old/new numbers: 2 3 safe stored name: RUNNING


**Further practice:** Rename `RUNNING` in the second enum: explain why name storage also needs a migration policy.

## Problem 21 — Build a name-based JSON codec with input validation

**Challenge.** Implement encode/decode functions that avoid implicit integer values, round-trip all enum members, and reject malformed payloads and unknown names.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 21

Parse untrusted JSON defensively, validate its shape, and convert using enum **names**. A stable name-based codec avoids dependence on generated ordinal values; it still needs an explicit policy when members are renamed or deleted.

In [22]:
@unique
class EventState21(Enum):
    QUEUED = auto()
    ACTIVE = auto()
    SUCCEEDED = auto()
    FAILED = auto()

def encode_state21(state: EventState21) -> str:
    if not isinstance(state, EventState21):
        raise TypeError("Expected EventState21")
    return json.dumps({"state": state.name}, separators=(",", ":"))

def decode_state21(raw: str) -> EventState21:
    obj = json.loads(raw)
    if not isinstance(obj, dict) or set(obj) != {"state"}:
        raise ValueError("Payload must contain only 'state'")
    if not isinstance(obj["state"], str):
        raise ValueError("State must be a string name")
    try:
        return EventState21[obj["state"]]
    except KeyError as exc:
        raise ValueError("Unknown state name") from exc

for state in EventState21:
    assert decode_state21(encode_state21(state)) is state
for invalid in ('{"state":"MISSING"}', '{"state":1}', '{}'):
    try:
        decode_state21(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Should reject: {invalid}")
print("Example JSON:", encode_state21(EventState21.ACTIVE))

Example JSON: {"state":"ACTIVE"}


**Further practice:** Add a migration map that accepts the deprecated name `IN_PROGRESS` as an alias for `ACTIVE` without adding an enum alias.

## Problem 22 — Apply enum members as state-machine transition keys

**Challenge.** Implement a validated workflow with transitions keyed by enum objects, not their numeric values. Reject illegal transitions explicitly.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 22

The generated numbers never enter business logic. The dictionary keys are actual enum members, making transitions readable, reviewable, and robust to insertion of new members.

In [23]:
@unique
class MachineState22(Enum):
    IDLE = auto()
    RUNNING = auto()
    PAUSED = auto()
    COMPLETE = auto()
    FAILED = auto()

TRANSITIONS22 = {
    MachineState22.IDLE: frozenset({MachineState22.RUNNING}),
    MachineState22.RUNNING: frozenset({
        MachineState22.PAUSED, MachineState22.COMPLETE, MachineState22.FAILED
    }),
    MachineState22.PAUSED: frozenset({
        MachineState22.RUNNING, MachineState22.FAILED
    }),
    MachineState22.COMPLETE: frozenset(),
    MachineState22.FAILED: frozenset(),
}

def transition22(current: MachineState22, target: MachineState22) -> MachineState22:
    if not isinstance(current, MachineState22) or not isinstance(target, MachineState22):
        raise TypeError("Both arguments must be MachineState22")
    if target not in TRANSITIONS22[current]:
        raise ValueError(f"Illegal transition: {current.name} -> {target.name}")
    return target

assert set(TRANSITIONS22) == set(MachineState22)
state22 = transition22(MachineState22.IDLE, MachineState22.RUNNING)
state22 = transition22(state22, MachineState22.PAUSED)
state22 = transition22(state22, MachineState22.RUNNING)
state22 = transition22(state22, MachineState22.COMPLETE)
assert state22 is MachineState22.COMPLETE
try:
    transition22(MachineState22.COMPLETE, MachineState22.RUNNING)
except ValueError as exc:
    print("Correctly rejected:", exc)
else:
    raise AssertionError("Terminal state should reject restart")

Correctly rejected: Illegal transition: COMPLETE -> RUNNING


**Further practice:** Add a `CANCELLED` state and update the transition table and exhaustiveness assertion.

## Problem 23 — Explicitly pin externally visible protocol codes

**Challenge.** Design a protocol enum where numbers are a documented contract and contrast it with a separate internal auto enum. Reject unknown values.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 23

Use **explicit** values whenever consumers depend on particular numbers. `auto()` is attractive for internal states whose numeric assignments are irrelevant; these domains should not share a design by accident.

In [24]:
@unique
class HttpLikeCode23(IntEnum):
    OK = 200
    BAD_REQUEST = 400
    NOT_FOUND = 404
    INTERNAL_ERROR = 500

@unique
class InternalPhase23(Enum):
    RECEIVED = auto()
    VALIDATED = auto()
    DISPATCHED = auto()

def parse_protocol_code23(raw: int) -> HttpLikeCode23:
    if type(raw) is not int:   # reject bool, which is an int subclass
        raise TypeError("Protocol code must be an integer")
    try:
        return HttpLikeCode23(raw)
    except ValueError as exc:
        raise ValueError(f"Unsupported protocol code: {raw}") from exc

assert parse_protocol_code23(404) is HttpLikeCode23.NOT_FOUND
assert InternalPhase23.RECEIVED.value == 1
for bad in (True, "404", 999):
    try:
        parse_protocol_code23(bad)
    except (TypeError, ValueError):
        pass
    else:
        raise AssertionError(f"Should reject {bad!r}")
print("Contract numeric value:", HttpLikeCode23.NOT_FOUND.value)

Contract numeric value: 404


**Further practice:** Design two additional protocol codes with fixed numbers and test round-trip conversion from integer.

## Problem 24 — Create an alias-aware inventory and regression checker

**Challenge.** Write a small library function that reports canonical values and aliases for arbitrary Enum classes; use it to assert the exact alias relationships of the source lesson.

**Success criteria.** Run the solution cell: its assertions must pass. Try solving before revealing/reading the code.

### Worked solution 24

The inventory bridges `Enum` iteration and `__members__`. It reveals accidental and intentional aliases without relying on implementation internals such as `_member_map_`. This exercise uses a deliberately mixed enum with a preceding maximum of 10.

In [25]:
def inventory24(enum_cls: type[Enum]) -> dict:
    canonical = [(member.name, member.value) for member in enum_cls]
    aliases = {
        declared_name: member.name
        for declared_name, member in enum_cls.__members__.items()
        if declared_name != member.name
    }
    return {
        "declared_names": list(enum_cls.__members__),
        "canonical": canonical,
        "aliases": aliases,
        "canonical_count": len(enum_cls),
    }

class AliasStudy24(Enum):
    GREEN = 1
    ALSO_GREEN = 1
    RED = 10
    ALSO_RED = 10
    BLUE = auto()

report24 = inventory24(AliasStudy24)
assert report24["aliases"] == {
    "ALSO_GREEN": "GREEN", "ALSO_RED": "RED"
}
assert report24["canonical_count"] == 3
assert report24["declared_names"] == [
    "GREEN", "ALSO_GREEN", "RED", "ALSO_RED", "BLUE"
]
assert AliasStudy24.BLUE.value == 11
print(json.dumps(report24, indent=2))

{
  "declared_names": [
    "GREEN",
    "ALSO_GREEN",
    "RED",
    "ALSO_RED",
    "BLUE"
  ],
  "canonical": [
    [
      "GREEN",
      1
    ],
    [
      "RED",
      10
    ],
    [
      "BLUE",
      11
    ]
  ],
  "aliases": {
    "ALSO_GREEN": "GREEN",
    "ALSO_RED": "RED"
  },
  "canonical_count": 3
}


**Further practice:** Change the report to emit a `canonical_name -> list of all declared names` mapping.

---
# Extra worked recipes and edge cases

These additional executable examples consolidate subtle points across the 24 problems. Run them after the worked solutions.

### Recipe A — A generator based on distinct values, not declaration count

If declarations can include aliases, `count` grows for aliases too. A distinct-value-aware algorithm must explicitly decide how to handle them. Here, value `1` has two names, but the next generated value is the next distinct integer.

In [26]:
class DistinctAwareA(Enum):
    @staticmethod
    def _generate_next_value_(name, start, count, last_values):
        return max(set(last_values), default=0) + 1

    PRIMARY = 1
    SYNONYM = 1
    NEXT = auto()
    FINAL = auto()

assert [m.value for m in DistinctAwareA] == [1, 2, 3]
assert DistinctAwareA.SYNONYM is DistinctAwareA.PRIMARY
print("Declared names:", list(DistinctAwareA.__members__))

Declared names: ['PRIMARY', 'SYNONYM', 'NEXT', 'FINAL']


### Recipe B — Inspect the Python version without depending on a changed default

The ordinary next-value algorithm for a deliberately mixed enum changed in Python 3.13. Rather than creating an exercise that expects the same result under all historical versions, write a custom generator with an explicit rule and test that rule.

In [27]:
@unique
class ExplicitRuleB(Enum):
    @staticmethod
    def _generate_next_value_(name, start, count, last_values):
        return max(last_values, default=0) + 1

    FIRST = 10
    SECOND = 5
    THIRD = auto()

assert ExplicitRuleB.THIRD.value == 11
print("Current interpreter:", sys.version.split()[0],
      "explicitly specified next value:", ExplicitRuleB.THIRD.value)

Current interpreter: 3.13.7 explicitly specified next value: 11


### Recipe C — Test JSON behavior of `StrEnum` vs plain `Enum`

`StrEnum` is a `str` subclass. Ordinary `Enum` with a `str` value is not itself a string.

In [28]:
assert json.dumps(WireState11.READY) == '"ready"'
try:
    json.dumps(PlainStringState11.READY)
except TypeError:
    print("Plain Enum members need an explicit encoding")
else:
    raise AssertionError("Expected plain Enum member to be non-JSON-serializable")
assert json.dumps(PlainStringState11.READY.value) == '"ready"'
assert json.loads(json.dumps(WireState11.DONE)) == "done"

Plain Enum members need an explicit encoding


### Recipe D — Stable text labels should not be confused with stable protocol values

Treat user-facing display labels, member names, and public API values as separate concepts. This explicit wire enum intentionally does **not** use `auto()`.

In [29]:
@unique
class PublicEventD(StrEnum):
    TASK_STARTED = "task.started.v1"
    TASK_FINISHED = "task.finished.v1"

DISPLAY_LABELS_D = {
    PublicEventD.TASK_STARTED: "Task has begun",
    PublicEventD.TASK_FINISHED: "Task completed successfully",
}
assert DISPLAY_LABELS_D[PublicEventD.TASK_STARTED] == "Task has begun"
assert PublicEventD.TASK_STARTED.value == "task.started.v1"
assert PublicEventD.TASK_STARTED.name == "TASK_STARTED"
print(PublicEventD.TASK_STARTED.name, PublicEventD.TASK_STARTED.value,
      DISPLAY_LABELS_D[PublicEventD.TASK_STARTED])

TASK_STARTED task.started.v1 Task has begun


### Recipe E — Alias-safe counts and invariants

When aliases are intentional, comparing `len(MyEnum)` with `len(MyEnum.__members__)` is supposed to show a difference.

In [30]:
assert len(AliasStudy24) == 3
assert len(AliasStudy24.__members__) == 5
assert sum(name != member.name for name, member in AliasStudy24.__members__.items()) == 2
assert all(member in AliasStudy24 for member in AliasStudy24)
print("canonical / declared / aliases:",
      len(AliasStudy24), len(AliasStudy24.__members__),
      len(AliasStudy24.__members__) - len(AliasStudy24))

canonical / declared / aliases: 3 5 2


### Recipe F — Property-based thinking with zero third-party dependencies

Instead of checking only one example, systematically check every generated member for uniqueness, type, round-trip lookup, and correct ordering.

In [31]:
def assert_unique_integer_enum_f(enum_cls: type[Enum]):
    members = list(enum_cls)
    values = [m.value for m in members]
    assert members, "At least one member is required"
    assert all(type(value) is int for value in values)
    assert len(values) == len(set(values))
    assert len(enum_cls.__members__) == len(members)
    assert all(enum_cls[member.name] is member for member in members)
    assert all(enum_cls(member.value) is member for member in members)

for cls in (WorkflowStage01, Fixed03, PowersOfThree05, SafeMixed06,
            Priority07, LegacyCode12, Continuous15, HttpLikeCode23):
    assert_unique_integer_enum_f(cls)
print("Property checks passed for 8 integer-valued enum classes")

Property checks passed for 8 integer-valued enum classes


### Recipe G — A tiny structured code-review checklist

Mark each design decision before accepting a new enum in production. These are questions, not automatic one-size-fits-all rules.

1. Are values internal or are they stored, transmitted, or used by external clients?
2. Is member identity the appropriate basis for business logic?
3. Are aliases permitted? If not, add `@unique`.
4. Does a custom generator appear before all members and clearly define collision handling?
5. Is a bitmask really needed? If yes, consider `Flag` or `IntFlag`.
6. Are serializers explicit about accepting names, values, deprecated names, and unknown input?
7. Would member insertion, reordering, or renaming break consumers?
8. Do tests prove uniqueness, desired values, round trips, and expected rejection behavior?

---
# Final integrated verification

The next cell independently checks the key results from every section. It deliberately uses `assert` rather than trusting printed output. If a cell above fails, resolve the failure before treating the notebook as complete.

In [32]:
CHECKS_FINAL = {
    "default enum auto": [x.value for x in WorkflowStage01] == [1, 2, 3],
    "accidental alias": Collision02.FIRST is Collision02.SECOND,
    "audit alias count": audit04[0][2] == 2,
    "custom powers": PowersOfThree05.FOURTH.value == 81,
    "mixed explicit generator": SafeMixed06.HIGHEST.value == 22,
    "functional start": Priority07.LOW.value == 10,
    "string generator": HumanState08.IN_PROGRESS.value == "In Progress",
    "slug generator": GoodSlug09.API_READY.value == "api-ready",
    "reusable base": ResourceKind10.FILE.value == "file",
    "StrEnum auto": WireState11.DONE.value == "done",
    "IntEnum arithmetic": type(LegacyCode12.SUCCESS + LegacyCode12.RETRY) is int,
    "flag bits": [x.value for x in Permission13] == [1, 2, 4, 8],
    "flag composite": NamedPermission14.READ_WRITE.value == 3,
    "continuous verification": Continuous15.THIRD.value == 7,
    "named flag verification": ValidMask16.READ_EXECUTE.value == 5,
    "random reproducibility": [m.value for m in r1_17] == [m.value for m in r2_17],
    "opaque values": JobState19.QUEUED.value is not JobState19.RUNNING.value,
    "versioned name storage": restored20 is VersionTwo20.RUNNING,
    "JSON round trip": decode_state21(encode_state21(EventState21.FAILED)) is EventState21.FAILED,
    "state machine": state22 is MachineState22.COMPLETE,
    "explicit wire protocol": parse_protocol_code23(200) is HttpLikeCode23.OK,
    "alias inventory": len(report24["aliases"]) == 2,
}
assert all(CHECKS_FINAL.values()), CHECKS_FINAL
print(f"PASS: {sum(CHECKS_FINAL.values())}/{len(CHECKS_FINAL)} integrated assertions")
print("All 24 problems, seven recipes, and integrated checks completed.")

PASS: 22/22 integrated assertions
All 24 problems, seven recipes, and integrated checks completed.


## Further reading and takeaways

The supplied lesson introduces basic `auto()`, mixed explicit values, custom `_generate_next_value_`, reuse through member-free base enums, object sentinels, and aliases. This workbook extends those topics with modern enum variants, strict validation, version compatibility, reproducible bounded random examples, defensive JSON parsing, workflow transition design, and tests.

**Remember:** prefer comparing enum members; use explicit values for stable external contracts; inspect `__members__` for aliases; use `@unique` when aliases are forbidden; implement explicit generators when their exact behavior matters; and run the verification cells after edits.

Documentation: [Python enum reference](https://docs.python.org/3/library/enum.html), [Enum HOWTO](https://docs.python.org/3/howto/enum.html).